<a href="https://colab.research.google.com/github/Epot12/Lab_XAI/blob/master/KERAS_ORIGINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from keras.models import Sequential, save_model
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from joblib import dump

orbit_to_remove = []
with open('/content/drive/MyDrive/orbit_to_remove.txt') as file:
    for line in file:
        orbit_to_remove.append(float(line))
df = pd.read_csv("/content/drive/MyDrive/MARSIS_historical_dataset.csv", sep=";")

frequency_to_keep = 4000000.0
df = df[df['FM_data_frequency'] == frequency_to_keep]
df = df[~df.FM_data_orbit_number.isin(orbit_to_remove)]
df['FM_data_solar_longitude_cos'] = np.cos(df['FM_data_solar_longitude'])
df['FM_data_solar_longitude_sin'] = np.sin(df['FM_data_solar_longitude'])

X = df.drop(columns=['FM_data_ephemeris_time', 'FM_data_F10_7_index', 'FM_data_frequency',
                     'FM_data_median_corrected_echo_power', 'FM_data_orbit_number', 'FM_data_peak_corrected_echo_power',
                     'FM_data_peak_distorted_echo_power', 'FM_data_peak_simulated_echo_power', 'FM_data_solar_longitude'])
col_names = X.columns.tolist()
X = X.to_numpy()

y = df['FM_data_peak_distorted_echo_power'].to_numpy()

kf = KFold(n_splits=10, shuffle=False)
kf.get_n_splits(X)

fold = -1
y_all_pred = np.zeros(y.shape)
f = open("MAE_nn_chrono.txt", "w")
t1 = time.time()
for train_index, test_index in kf.split(X):
    fold = fold + 1
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    f.write('TRAIN: ' + str(train_index)+'\n')
    f.write('TEST: ' + str(test_index)+'\n')
    dump(scaler, f'scaler_chrono_{fold}.save')

    # Neural Network
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
    model = Sequential()
    model.add(Dense(800, input_dim=X.shape[1], activation='relu'))  # 300
    model.add(Dropout(0.5))
    model.add(Dense(400, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(200, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(100, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='relu'))
    model.compile(loss='mse', optimizer='adam', metrics=['mean_absolute_error'])
    model.fit(X_train, y_train, validation_split=0.1, epochs=1000, callbacks=[es])
    save_model(model, f'nn_chrono_model_{fold}.keras')
    # evaluate the model
    _, train_mse = model.evaluate(X_train, y_train, verbose=0)
    _, test_mse = model.evaluate(X_test, y_test, verbose=0)
    print('Train: %.3f, Test: %.3f' % (train_mse, test_mse))
    t2 = time.time()

    y_pred = model.predict(X_test).flatten()
    f.write(f'MAE = {mean_absolute_error(y_test,y_pred)}\n')
    f.write(f'MAPE = {mean_absolute_percentage_error(y_test, y_pred)}\n')
    f.write(f'MSE = {mean_squared_error(y_test, y_pred)}\n')
    f.write(f'Execution time = {t2 - t1}\n')
    for i in range(len(test_index)):
        y_all_pred[test_index[i]] = y_pred[i]

t3 = time.time()
f.write(f'\nGlobal MAE = {mean_absolute_error(y, y_all_pred)}')
f.write(f'\nGlobal MAPE = {mean_absolute_percentage_error(y, y_all_pred)}')
f.write(f'\nGlobal MSE = {mean_squared_error(y, y_all_pred)}')
f.write(f'\nTime = {t3-t1}')
f.close()
np.savetxt("nn_y_all_pred_chrono.txt", y_all_pred)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 126s 2ms/step - loss: 14.0833 - mean_absolute_error: 2.8671 - val_loss: 12.3694 - val_mean_absolute_error: 2.7785
Epoch 2/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 99s 2ms/step - loss: 10.0930 - mean_absolute_error: 2.4600 - val_loss: 12.2874 - val_mean_absolute_error: 2.7633
Epoch 3/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 99s 2ms/step - loss: 9.6028 - mean_absolute_error: 2.3954 - val_loss: 11.6674 - val_mean_absolute_error: 2.5964
Epoch 4/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 99s 2ms/step - loss: 9.3749 - mean_absolute_error: 2.3618 - val_loss: 11.9952 - val_mean_absolute_error: 2.6678
Epoch 5/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 98s 2ms/step - loss: 9.2487 - mean_absolute_error: 2.3440 - val_loss: 12.0887 - val_mean_absolute_error: 2.7632
Epoch 6/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 99s 2ms/step - loss: 9.1667 - mean_absolute_error: 2.3310 - val_loss: 10.6878 - val_mean_absolute_error: 2.4562
Epoch 7/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 99s

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 117s 2ms/step - loss: 13.6478 - mean_absolute_error: 2.8024 - val_loss: 11.1828 - val_mean_absolute_error: 2.5167
Epoch 2/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 9.9769 - mean_absolute_error: 2.4361 - val_loss: 10.7239 - val_mean_absolute_error: 2.4883
Epoch 3/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 9.5113 - mean_absolute_error: 2.3752 - val_loss: 12.2983 - val_mean_absolute_error: 2.5955
Epoch 4/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 9.3150 - mean_absolute_error: 2.3465 - val_loss: 11.1918 - val_mean_absolute_error: 2.4967
Epoch 5/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 9.2090 - mean_absolute_error: 2.3317 - val_loss: 10.6678 - val_mean_absolute_error: 2.4555
Epoch 6/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 9.1537 - mean_absolute_error: 2.3233 - val_loss: 10.5577 - val_mean_absolute_error: 2.4426
Epoch 7/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

Epoch 1/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 111s 2ms/step - loss: 13.3807 - mean_absolute_error: 2.7824 - val_loss: 12.4256 - val_mean_absolute_error: 2.7092
Epoch 2/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 9.7847 - mean_absolute_error: 2.4197 - val_loss: 13.0769 - val_mean_absolute_error: 2.8088
Epoch 3/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 9.3267 - mean_absolute_error: 2.3578 - val_loss: 12.0066 - val_mean_absolute_error: 2.6729
Epoch 4/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 9.1080 - mean_absolute_error: 2.3261 - val_loss: 11.9414 - val_mean_absolute_error: 2.6079
Epoch 5/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 8.9929 - mean_absolute_error: 2.3088 - val_loss: 11.8887 - val_mean_absolute_error: 2.6098
Epoch 6/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 8.9255 - mean_absolute_error: 2.2988 - val_loss: 13.0976 - val_mean_absolute_error: 2.7901
Epoch 7/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 111s 2ms/step - loss: 13.7091 - mean_absolute_error: 2.7972 - val_loss: 11.1542 - val_mean_absolute_error: 2.5827
Epoch 2/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 10.1064 - mean_absolute_error: 2.4476 - val_loss: 14.2049 - val_mean_absolute_error: 2.9354
Epoch 3/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 9.5884 - mean_absolute_error: 2.3797 - val_loss: 13.1204 - val_mean_absolute_error: 2.7530
Epoch 4/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 9.3984 - mean_absolute_error: 2.3546 - val_loss: 12.6910 - val_mean_absolute_error: 2.7307
Epoch 5/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 9.2844 - mean_absolute_error: 2.3384 - val_loss: 13.3197 - val_mean_absolute_error: 2.8049
Epoch 6/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 9.2198 - mean_absolute_error: 2.3299 - val_loss: 13.1830 - val_mean_absolute_error: 2.7829
Epoch 7/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 112s 2ms/step - loss: 13.8947 - mean_absolute_error: 2.8215 - val_loss: 21.4277 - val_mean_absolute_error: 3.6987
Epoch 2/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 10.0776 - mean_absolute_error: 2.4470 - val_loss: 16.7993 - val_mean_absolute_error: 3.2192
Epoch 3/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 9.5628 - mean_absolute_error: 2.3785 - val_loss: 22.1795 - val_mean_absolute_error: 3.5964
Epoch 4/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 101s 2ms/step - loss: 9.4034 - mean_absolute_error: 2.3564 - val_loss: 19.4912 - val_mean_absolute_error: 3.3850
Epoch 5/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 9.3125 - mean_absolute_error: 2.3449 - val_loss: 20.4428 - val_mean_absolute_error: 3.5242
Epoch 6/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 100s 2ms/step - loss: 9.2733 - mean_absolute_error: 2.3384 - val_loss: 24.3780 - val_mean_absolute_error: 3.7511
Epoch 7/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━